<a href="https://colab.research.google.com/github/peremartra/Rearchitecting-LLMs/blob/main/CH07/CH07_NB_dataset_generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<table style="width:100%; border:none; background:none;">
  <tr style="border:none;">
    <td style="border:none; vertical-align:middle; text-align:left; width: 120px;">
      <a href="https://hubs.la/Q040tvsK0"><img src="https://raw.githubusercontent.com/peremartra/Rearchitecting-LLMs/main/Images/cover.png" width="100px" style="border-radius: 4px;"></a>
    </td>
    <td style="border:none; vertical-align:middle; text-align:left;">
      <p style="margin: 0; font-size: 14px;">
        Supplementary code for the <a href="https://hubs.la/Q040tvsK0">Rearchitecting LLMs</a> book by <a href="https://github.com/peremartra">Pere Martra</a>.<br>
        <br>
        Code repository: <a href="https://github.com/peremartra/Rearchitecting-LLMs">https://github.com/peremartra/Rearchitecting-LLMs</a>
      </p>
    </td>
  </tr>
</table>

#Rearchitecting LLMs
## Structural techniques for efficient models


### Chapter 7: Specialization Tuning.

[![LinkedIn](https://img.shields.io/badge/LinkedIn-0077B5?style=flat&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/pere-martra/) [![GitHub](https://img.shields.io/badge/GitHub-100000?style=flat&logo=github&logoColor=white)](https://github.com/peremartra) [![X](https://img.shields.io/badge/X-000000?style=flat&logo=x&logoColor=white)](https://x.com/PereMartra) [![Hugging Face](https://img.shields.io/badge/🤗%20Hugging%20Face-blue)](https://huggingface.co/oopere)

_____
Colab Environment: CPU

_____


# Chapter 7 — Clinical NER Dataset Generator

This notebook generates a synthetic clinical dataset for fine-tuning with QDoRA.
Each row contains a clinical note and its corresponding structured JSON label.

**Compatible with:** OpenAI, Anthropic Claude, Google Gemini (via LiteLLM)

**Output:** ~400 rows across 5 categories, uploaded to Hugging Face.

## 1. Install dependencies

In [1]:
!pip install -q litellm==1.82.6
!pip install -q datasets
!pip install -1 huggingface_hub



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.6/15.6 MB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 7.7 MB/s eta 0:00:00

Usage:   
  pip3 install [options] <requirement specifier> [package-index-options] ...
  pip3 install [options] -r <requirements file> [package-index-options] ...
  pip3 install [options] [-e] <vcs project url> ...
  pip3 install [options] [-e] <local project path> ...
  pip3 install [options] <archive url/path> ...

no such option: -1


In [ ]:
from google.colab import drive
from collections import Counter
import os
from google.colab import userdata

drive.mount("/content/drive")

CHECKPOINT_PATH = "/content/drive/MyDrive/RearchitectingLLMs/ch7_dataset_checkpoint.json"
# Crear el directorio si no existe
os.makedirs(os.path.dirname(CHECKPOINT_PATH), exist_ok=True)

def load_checkpoint() -> list[dict]:
    if os.path.exists(CHECKPOINT_PATH):
        with open(CHECKPOINT_PATH, "r") as f:
            data = json.load(f)
        print(f"Checkpoint found: {len(data)} samples loaded.")
        return data
    print("No checkpoint found. Starting from scratch.")
    return []

def save_checkpoint(samples: list[dict]):
    with open(CHECKPOINT_PATH, "w") as f:
        json.dump(samples, f)
        f.flush()
        os.fsync(f.fileno())

Mounted at /content/drive


## 2. Configuration

Set your API key and choose your model.

| Provider   | Model string example                  |
|------------|---------------------------------------|
| OpenAI     | `gpt-4o`                              |
| Anthropic  | `claude-sonnet-4-20250514`            |
| Gemini     | `gemini/gemini-2.0-flash`             |

In [ ]:
import os

# --- Choose your provider ---
MODEL = "anthropic/claude-sonnet-4-20250514"
# "gemini/gemini-3.1-pro-preview"  # Change this to your preferred model

# --- Set your API key ---
# For OpenAI:
# os.environ["OPENAI_API_KEY"] = "sk-..."

# For Anthropic:
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

from google.colab import userdata

# Acceder al secret de Colab
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

# --- Hugging Face token (for uploading the dataset) ---
HF_DATASET_NAME = "oopere/clinical-ner-qdora"  # Change to your HF username
HF_TOKEN = userdata.get("HF_TOKEN")
print(f"Model: {MODEL}")
print("Configuration ready.")

Model: anthropic/claude-sonnet-4-20250514
Configuration ready.


## 3. JSON Schema and Validation

Every generated sample must conform to this schema.
Fields with no information in the note must be `null`, never omitted.

In [ ]:
import json

SCHEMA_DESCRIPTION = """
{
  "patient_age": int or null,
  "symptoms": [list of normalized strings],
  "vital_signs": {
    "temperature": float or null,
    "heart_rate": int or null,
    "blood_pressure": string or null
  },
  "medications": [
    {"name": string, "dose": string, "frequency": string}
  ],
  "duration_days": int or null
}
"""

def validate_label(label: dict) -> tuple[bool, str]:
    """
    Validates that a label conforms to the required schema.
    Returns (True, "") if valid, (False, reason) if not.
    """
    # Top-level required keys
    required_keys = ["patient_age", "symptoms", "vital_signs", "medications", "duration_days"]
    for key in required_keys:
        if key not in label:
            return False, f"Missing top-level key: '{key}'"

    # patient_age: int or null
    if label["patient_age"] is not None and not isinstance(label["patient_age"], int):
        return False, f"'patient_age' must be int or null, got {type(label['patient_age'])}"

    # symptoms: list of strings
    if not isinstance(label["symptoms"], list):
        return False, "'symptoms' must be a list"
    if not all(isinstance(s, str) for s in label["symptoms"]):
        return False, "All items in 'symptoms' must be strings"

    # vital_signs: dict with 3 required keys
    if not isinstance(label["vital_signs"], dict):
        return False, "'vital_signs' must be a dict"
    vital_keys = ["temperature", "heart_rate", "blood_pressure"]
    for key in vital_keys:
        if key not in label["vital_signs"]:
            return False, f"Missing key in 'vital_signs': '{key}'"
    if label["vital_signs"]["temperature"] is not None and not isinstance(label["vital_signs"]["temperature"], (int, float)):
        return False, "'temperature' must be float or null"
    if label["vital_signs"]["heart_rate"] is not None and not isinstance(label["vital_signs"]["heart_rate"], int):
        return False, "'heart_rate' must be int or null"
    if label["vital_signs"]["blood_pressure"] is not None and not isinstance(label["vital_signs"]["blood_pressure"], str):
        return False, "'blood_pressure' must be string or null"

    # medications: list of dicts with name, dose, frequency
    if not isinstance(label["medications"], list):
        return False, "'medications' must be a list"
    for i, med in enumerate(label["medications"]):
        if not isinstance(med, dict):
            return False, f"medication[{i}] must be a dict"
        for med_key in ["name", "dose", "frequency"]:
            if med_key not in med:
                return False, f"medication[{i}] missing key: '{med_key}'"
            if not isinstance(med[med_key], str):
                return False, f"medication[{i}]['{med_key}'] must be a string"

    # duration_days: int or null
    if label["duration_days"] is not None and not isinstance(label["duration_days"], int):
        return False, f"'duration_days' must be int or null, got {type(label['duration_days'])}"

    return True, ""


def parse_and_validate(raw_response: str) -> tuple[dict | None, str]:
    """
    Parses the LLM response and validates its structure.
    Returns (sample_dict, "") if valid, (None, reason) if not.
    """
    # Strip markdown fences if present
    text = raw_response.strip()
    if text.startswith("```"):
        text = text.split("```")[1]
        if text.startswith("json"):
            text = text[4:]
        text = text.strip()

    # Parse JSON
    try:
        sample = json.loads(text)
    except json.JSONDecodeError as e:
        return None, f"JSON parse error: {e}"

    # Check top-level structure
    if "note" not in sample:
        return None, "Missing top-level key: 'note'"
    if "label" not in sample:
        return None, "Missing top-level key: 'label'"
    if not isinstance(sample["note"], str) or not sample["note"].strip():
        return None, "'note' must be a non-empty string"

    # Validate label schema
    is_valid, reason = validate_label(sample["label"])
    if not is_valid:
        return None, f"Label validation failed: {reason}"

    return sample, ""


print("Schema and validation functions ready.")

Schema and validation functions ready.


## 4. Prompt Templates

Each category has a specific user prompt that guides the LLM
to generate the appropriate type of clinical note.

In [ ]:
SYSTEM_PROMPT = f"""You are a synthetic clinical data generator.
Your task is to generate realistic clinical notes and their corresponding structured labels.

OUTPUT FORMAT: Respond exclusively with a single JSON object. No markdown, no explanations.
The object must have exactly two keys:
- "note": a clinical note string
- "label": a structured JSON object following this schema exactly:

{SCHEMA_DESCRIPTION}

STRICT RULES:
1. All 5 top-level keys must always be present in the label.
2. All 3 keys inside vital_signs must always be present.
3. If information is not present in the note, use null. Never omit a field.
4. In the label, always normalize clinical terms to plain English.
   Examples: SOB -> shortness of breath, HR -> heart rate, c/o -> complains of
5. patient_age must be an integer, not a string.
6. temperature must be a float (e.g. 38.5), not a string.
7. Each medication must have name, dose, and frequency as strings.
   If dose or frequency are not mentioned, use null is NOT allowed here: use "not specified".
"""

CATEGORY_PROMPTS = {
    "clean": (
        "Generate a clean, complete clinical note. "
        "Vary the condition, age (20-85), and sex across generations. "
        "Use different conditions: UTI, diabetes, hypertension, fracture, "
        "gastroenteritis, migraine, asthma, skin infection, anxiety, COPD. "
        "In approximately 30% of cases, omit one or two fields from the note "
        "(no vitals recorded, no duration mentioned, no medication prescribed yet). "
        "Use null for missing fields in the label."
    ),
    "abbreviations": (
        "Generate a clinical note using medical abbreviations: "
        "pt, yo, c/o, SOB, DOE, HR, BP, Temp, Dx, Rx, q8h, qd, bid, tid, PRN, IV, PO, stat, hx, sx. "
        "Vary the condition and patient demographics. "
        "In approximately 30% of cases, the note should be incomplete: "
        "no vitals taken yet, or duration not recorded, or medication pending. "
        "Use null for missing fields in the label. "
        "Normalize all abbreviated terms in the label."
    ),
    "implicit": (
        "Generate a clinical note where symptoms are described indirectly. "
        "Examples: 'can't climb stairs without stopping' instead of 'dyspnea', "
        "'wakes up drenched at night' instead of 'night sweats', "
        "'food has no taste' instead of 'ageusia', "
        "'heart pounding out of chest' instead of 'palpitations'. "
        "Vary the condition and demographics. "
        "In approximately 30% of cases, some fields are absent from the note. "
        "Use null for missing fields. Use correct normalized clinical terms in the label."
    ),
    "typos": (
        "Generate a clinical note with realistic typing errors, informal language, "
        "and incomplete sentences as written in a hurry. "
        "Include: misspellings, missing punctuation, lowercase, run-on sentences, word omissions. "
        "Vary the condition and patient demographics significantly. "
        "In approximately 40% of cases, the note is incomplete: "
        "vitals not yet taken, duration vague or absent, medication not yet decided. "
        "Use null for missing fields. The label must be clean and normalized."
    ),
    "irrelevant": (
        "Generate a clinical note with irrelevant or distracting information mixed in. "
        "Include some of: references to previous visits, administrative comments, "
        "physician opinions, social context, insurance notes, scheduling info. "
        "Vary the condition and patient demographics. "
        "In approximately 40% of cases, some clinical fields are genuinely absent "
        "(not just buried in irrelevant text). Use null for missing fields. "
        "The label must contain only clinically relevant extracted information."
    ),
}

# Number of samples to generate per category
SAMPLES_PER_CATEGORY = {
    "clean":         100,
    "abbreviations":  80,
    "implicit":       80,
    "typos":          70,
    "irrelevant":     70,
}

TOTAL = sum(SAMPLES_PER_CATEGORY.values())
print(f"Total samples to generate: {TOTAL}")
for cat, n in SAMPLES_PER_CATEGORY.items():
    print(f"  {cat}: {n}")

Total samples to generate: 400
  clean: 100
  abbreviations: 80
  implicit: 80
  typos: 70
  irrelevant: 70


## 5. Generation Loop

For each category, we call the LLM once per sample.
Failed validations are automatically retried up to `MAX_RETRIES` times.

In [ ]:
import time
from litellm import completion

MAX_RETRIES = 3
TEMPERATURE = 0.9  # High temperature for variety


def generate_sample(category: str, attempt: int = 1) -> dict | None:
    """
    Generates a single validated sample for the given category.
    Returns the sample dict or None if all retries fail.
    """
    user_prompt = CATEGORY_PROMPTS[category]

    try:
        response = completion(
            model=MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": user_prompt},
            ],
            temperature=TEMPERATURE,
            max_tokens=1024,
        )
        raw = response.choices[0].message.content
    except Exception as e:
        print(f"      API error on attempt {attempt}: {e}")
        return None

    sample, reason = parse_and_validate(raw)

    if sample is not None:
        # Add metadata
        sample["category"] = category
        return sample
    else:
        print(f"      Validation failed (attempt {attempt}): {reason}")
        return None


def generate_category(category: str, n: int) -> list[dict]:
    """
    Generates n validated samples for a category, retrying failures.
    """
    samples = []
    attempts_total = 0

    print(f"\nGenerating '{category}' ({n} samples)...")

    while len(samples) < n:
        remaining = n - len(samples)
        print(f"  [{len(samples)}/{n}] {remaining} remaining...", end="\r")

        sample = None
        for attempt in range(1, MAX_RETRIES + 1):
            sample = generate_sample(category, attempt)
            attempts_total += 1
            if sample is not None:
                break
            time.sleep(1)  # Brief pause before retry

        if sample is not None:
            samples.append(sample)
        else:
            print(f"\n  WARNING: Could not generate a valid sample after {MAX_RETRIES} attempts. Skipping.")

        # Small delay to avoid rate limits
        time.sleep(0.3)

    print(f"  [{n}/{n}] Done. (Total API calls: {attempts_total})          ")
    return samples


print("Generation functions ready.")

Generation functions ready.


## 6. Run Generation

This cell generates all samples. Expect ~10-15 minutes depending on the model and API speed.

In [ ]:
all_samples = load_checkpoint()

# Count already generated per category
already_done = Counter(s["category"] for s in all_samples)

for category, n in SAMPLES_PER_CATEGORY.items():
    done = already_done.get(category, 0)
    remaining = n - done
    if remaining <= 0:
        print(f"'{category}': already complete ({done}/{n}), skipping.")
        continue

    print(f"\nGenerating '{category}' ({done}/{n} already done, {remaining} remaining)...")
    count = done

    while count < n:
        print(f"  [{count}/{n}]...", end="\r")
        sample = None
        for attempt in range(1, MAX_RETRIES + 1):
            sample = generate_sample(category, attempt)
            if sample is not None:
                break
            time.sleep(1)

        if sample is not None:
            all_samples.append(sample)
            save_checkpoint(all_samples)  # Save immediately after each valid sample
            count += 1
        else:
            print(f"\n  WARNING: Skipping after {MAX_RETRIES} failed attempts.")

        time.sleep(0.3)

    print(f"  [{n}/{n}] '{category}' complete.          ")

print(f"\nTotal samples: {len(all_samples)} / {TOTAL}")

Checkpoint found: 400 samples loaded.
'clean': already complete (100/100), skipping.
'abbreviations': already complete (80/80), skipping.
'implicit': already complete (80/80), skipping.
'typos': already complete (70/70), skipping.
'irrelevant': already complete (70/70), skipping.

Total samples: 400 / 400


In [ ]:
import os
print(os.path.exists(CHECKPOINT_PATH))
print(os.path.getsize(CHECKPOINT_PATH) if os.path.exists(CHECKPOINT_PATH) else "File not found")

True
358148


## 7. Inspect Results

Quick check of distribution and a sample from each category.

In [ ]:
from collections import Counter

# Distribution
distribution = Counter(s["category"] for s in all_samples)
print("Category distribution:")
for cat, count in distribution.items():
    print(f"  {cat}: {count}")

print()

# One example per category
seen = set()
for sample in all_samples:
    cat = sample["category"]
    if cat not in seen:
        seen.add(cat)
        print(f"{'='*70}")
        print(f"CATEGORY: {cat}")
        print(f"NOTE: {sample['note']}")
        print(f"LABEL: {json.dumps(sample['label'], indent=2)}")
        print()

Category distribution:
  clean: 100
  abbreviations: 80
  implicit: 80
  typos: 70
  irrelevant: 70

CATEGORY: clean
NOTE: HPI: 42-year-old male presents to the urgent care clinic complaining of a persistent productive cough and intermittent chest tightness. Patient states symptoms began 5 days ago following a mild upper respiratory infection. He reports subjective fevers at home and increased fatigue. Physical Exam: Patient is alert and oriented. Vital signs recorded as follows: Temp 101.2 F, HR 94 bpm, BP 132/88 mmHg. Lung auscultation reveals mild expiratory wheezing and rhonchi in the lower right lobe. Assessment: Community-acquired pneumonia. Plan: Start Amoxicillin 500mg three times daily for 7 days. Advised to increase fluid intake and return if symptoms worsen.
LABEL: {
  "patient_age": 42,
  "symptoms": [
    "productive cough",
    "chest tightness",
    "fever",
    "fatigue",
    "wheezing",
    "rhonchi"
  ],
  "vital_signs": {
    "temperature": 101.2,
    "heart_rate": 9

## 8. Upload to Hugging Face

We upload the dataset to Hugging Face Hub with a dataset card
that documents the generation process transparently.

In [ ]:
from datasets import Dataset
from huggingface_hub import HfApi

# Flatten for HF Dataset: note, label (as JSON string), category
hf_records = [
    {
        "note":     s["note"],
        "label":    json.dumps(s["label"]),  # Store as JSON string for portability
        "category": s["category"],
    }
    for s in all_samples
]

dataset = Dataset.from_list(hf_records)

# Split: 90% train, 10% test
split = dataset.train_test_split(test_size=0.1, seed=42)
print(f"Train: {len(split['train'])} samples")
print(f"Test:  {len(split['test'])} samples")

# Upload
split.push_to_hub(
    HF_DATASET_NAME,
    token=HF_TOKEN,
    private=False,
)

print(f"\nDataset uploaded to: https://huggingface.co/datasets/{HF_DATASET_NAME}")

## 9. Dataset Card

Upload a README with metadata and generation documentation.

In [ ]:
dataset_card = f"""---
license: mit
task_categories:
- token-classification
language:
- en
tags:
- clinical
- ner
- fine-tuning
- qdora
- synthetic
---

# Clinical NER Dataset for QDoRA Fine-Tuning

Companion dataset for Chapter 7 of [*Rearchitecting LLMs* (Manning Publications)](https://github.com/peremartra/optipfair).

[![linkedin-profile-banner-martra](https://cdn-uploads.huggingface.co/production/uploads/640f7924f2d7c41a1e9eced1/sa4ivCbm8kk6C9NAPmb-x.jpeg)](https://hubs.la/Q040tvsK0)

## Task
Extract structured clinical information from free-text medical notes into a strict JSON schema.

## How This Dataset Is Used in Chapter 7

This dataset is the foundation of a fine-tuning experiment designed to demonstrate
a concrete problem: without fine-tuning, a language model produces correct JSON only
when given a long, carefully crafted system prompt. Change one word, use a vague
instruction, or switch to a complex note with abbreviations, and the model invents
its own schema. The behavior is fragile and unpredictable in production.

Chapter 7 uses this dataset to fine-tune Qwen3-1.7B with QDoRA, teaching the model
to respect a strict JSON contract regardless of how the instruction is phrased.
The five note categories (clean, abbreviations, implicit, typos, irrelevant) are
designed to force generalization: the model cannot memorize, it must learn the
extraction pattern across diverse clinical language.

## Schema
```json
{SCHEMA_DESCRIPTION.strip()}
```

## Dataset Construction
This dataset was synthetically generated a combination of diferent LLMs via LiteLLM.
All samples were automatically validated against the schema before inclusion.
No real patient data was used.

## Categories ({TOTAL} total samples)
| Category | Samples | Description |
|---|---|---|
| clean | 100 | Well-structured professional notes |
| abbreviations | 80 | Notes with standard medical abbreviations |
| implicit | 80 | Symptoms described indirectly |
| typos | 70 | Notes with typing errors and informal language |
| irrelevant | 70 | Notes with distracting irrelevant context |

## Fields
- `note`: free-text clinical note
- `label`: JSON string with the structured extraction
- `category`: generation category (for analysis)

## Splits
- Train: 90%
- Test: 10%
"""

api = HfApi()
api.upload_file(
    path_or_fileobj=dataset_card.encode(),
    path_in_repo="README.md",
    repo_id=HF_DATASET_NAME,
    repo_type="dataset",
    token=HF_TOKEN,
)

print("Dataset card uploaded.")
print(f"\nDone! Visit: https://huggingface.co/datasets/{HF_DATASET_NAME}")

Dataset card uploaded.

Done! Visit: https://huggingface.co/datasets/oopere/clinical-ner-qdora
